# 4.12 — Principal Component Analysis

Principal Component Analysis (PCA) finds directions in centered data where the points vary the most, then uses those directions as a lower-dimensional coordinate system. In this lesson, you will build PCA from raw NumPy operations: center the data, measure covariance, extract orthogonal directions, project into scores, reconstruct from a bottleneck, and audit when scaling or rank choices change the story.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build Principal Component Analysis one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is made visible so PCA is not just a call to a library. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays + linear algebra for centering, covariance, eigenvectors, and SVD.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for toy data and any random examples.

### 1. PCA starts by centering the cloud

PCA is about **variation around the average point**, not the absolute location of the data in the coordinate system. If every point is shifted 100 units to the right, the main direction of spread should not change. Centering subtracts each column mean so the data cloud has mean zero and every later dot product measures deviations from the baseline.

In [ ]:
X_w = np.array([[1., 2.],
                [2., 1.],
                [3., 4.],
                [4., 3.]])  # four points in two measured coordinates.
mean_w = X_w.mean(axis=0)  # one mean per feature/column.
Xc_w = X_w - mean_w  # centered data: deviations from the feature means.

print("column means:", mean_w)
print("centered data:\n", Xc_w)

assert np.allclose(mean_w, [2.5, 2.5])

▶ What you'll see: the original feature means are `[2.5, 2.5]`, and centered rows now sum to zero by column.

In [ ]:
plt.figure(figsize=(4.4, 3.6))
plt.scatter(X_w[:, 0], X_w[:, 1], s=80, label="raw points", color="steelblue")
plt.scatter(Xc_w[:, 0], Xc_w[:, 1], s=80, label="centered points", color="darkorange")
plt.axhline(0, color="gray", linewidth=0.8); plt.axvline(0, color="gray", linewidth=0.8)
plt.title("1: centering moves the cloud to the origin")
plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.legend(); plt.show()

▶ What you'll see: the orange centered cloud has the same shape as the blue raw cloud, but its average is at the origin.

*Why it's done this way:* PCA chooses directions by maximizing squared projected deviations. Without centering, the origin itself becomes part of the story, so a large offset can masquerade as meaningful variance. Subtracting the column mean makes the objective depend on spread, not where someone placed the axes.

### 2. Covariance measures which directions vary together

After centering, PCA needs a compact summary of spread. The covariance matrix is that summary: diagonal entries measure each feature's variance, and off-diagonal entries measure whether features rise and fall together. For centered data, the sample covariance is $C=X_c^\top X_c/(m-1)$.

In [ ]:
m_w = Xc_w.shape[0]  # number of examples.
C_w = (Xc_w.T @ Xc_w) / (m_w - 1)  # sample covariance matrix.

print("Xc.T @ Xc:\n", Xc_w.T @ Xc_w)
print("covariance C:\n", np.round(C_w, 3))

assert np.allclose(C_w, [[1.6666667, 1.0], [1.0, 1.6666667]])

▶ What you'll see: both features have the same variance, and the positive off-diagonal says they tend to increase together.

In [ ]:
plt.figure(figsize=(4, 3.4))
plt.imshow(C_w, cmap="coolwarm", vmin=-2, vmax=2)
plt.colorbar(label="covariance")
plt.xticks([0, 1], ["feature 0", "feature 1"]); plt.yticks([0, 1], ["feature 0", "feature 1"])
plt.title("2: covariance summarizes spread"); plt.show()

▶ What you'll see: positive diagonal and off-diagonal values, meaning both individual spread and shared movement are present.

*Why it's done this way:* The expression $v^\top C v$ equals the variance of the data after projecting onto direction $v$. So once we have $C$, choosing a principal component becomes a precise optimization problem: find the unit vector $v$ that makes $v^\top C v$ as large as possible.

### 3. Eigenvectors are the principal directions

A principal component is a unit direction that keeps as much variance as possible. For a symmetric covariance matrix, the best directions are its eigenvectors, and the variance captured by each direction is its eigenvalue. NumPy returns them in arbitrary order, so we sort from largest eigenvalue to smallest.

In [ ]:
evals_w, evecs_w = np.linalg.eigh(C_w)  # eigh is for symmetric matrices like covariance.
order_w = np.argsort(evals_w)[::-1]  # descending variance order.
evals_w = evals_w[order_w]
evecs_w = evecs_w[:, order_w]

print("eigenvalues:", np.round(evals_w, 3))
print("eigenvectors:\n", np.round(evecs_w, 3))

assert np.allclose(np.round(evals_w, 3), [2.667, 0.667])

▶ What you'll see: the first direction captures four times as much variance as the second.

In [ ]:
plt.figure(figsize=(4.4, 3.8))
plt.scatter(Xc_w[:, 0], Xc_w[:, 1], s=80, color="steelblue")
for j_w, color_w in enumerate(["crimson", "seagreen"]):
    vec_w = evecs_w[:, j_w] * np.sqrt(evals_w[j_w])
    plt.arrow(0, 0, vec_w[0], vec_w[1], head_width=0.08, color=color_w, length_includes_head=True)
    plt.text(vec_w[0] * 1.08, vec_w[1] * 1.08, f"PC{j_w+1}", color=color_w)
plt.axhline(0, color="gray", linewidth=0.8); plt.axvline(0, color="gray", linewidth=0.8)
plt.axis("equal"); plt.title("3: eigenvectors point along spread"); plt.show()

▶ What you'll see: PC1 lies along the long diagonal of the centered cloud, while PC2 is perpendicular.

*Why it's done this way:* The eigenvector equation $Cv=\lambda v$ says that multiplying by covariance stretches the direction without rotating it. Those no-rotation directions are exactly the natural axes of the data cloud, and sorting by $\lambda$ keeps the axis with maximum projected variance first.

### 4. SVD gives the same PCA directions from the data matrix

PCA is often implemented with the singular value decomposition $X_c=U\Sigma V^\top$. The right singular vectors $V$ are the principal directions, and singular values encode variance through $\sigma_j^2/(m-1)$. This is the formula from the lesson content: keep $V_r$, then compute $Z=X_cV_r$.

In [ ]:
U_w, s_w, Vt_w = np.linalg.svd(Xc_w, full_matrices=False)  # factor centered data directly.
V_w = Vt_w.T  # columns are right singular vectors / PCA directions.
variance_from_svd_w = (s_w ** 2) / (m_w - 1)

print("singular values:", np.round(s_w, 3))
print("variance from SVD:", np.round(variance_from_svd_w, 3))

assert np.allclose(np.round(s_w, 3), [2.828, 1.414])

▶ What you'll see: singular values `[2.828, 1.414]`, whose squared values are energies `[8, 2]`.

In [ ]:
alignment_w = np.abs(np.sum(V_w * evecs_w, axis=0))  # sign may flip, absolute alignment should be 1.

print("|SVD direction · eigen direction|:", np.round(alignment_w, 3))

assert np.allclose(np.round(alignment_w, 3), [1.0, 1.0])

▶ What you'll see: SVD and covariance eigenvectors agree up to harmless sign flips.

*Why it's done this way:* Forming covariance squares the condition number and can be less stable for large data. SVD works directly on centered data while still returning the same right-direction geometry, so it is the standard numerical route to PCA.

### 5. Projection creates lower-dimensional scores

Once directions are known, PCA coordinates are just projections: $Z=X_cV_r$. If we keep only the first component, every two-dimensional point becomes one number: its signed coordinate along the high-variance axis. This is compression, but it is also a new representation for downstream models.

In [ ]:
V1_w = V_w[:, :1]  # keep the first principal direction only.
Z1_w = Xc_w @ V1_w  # one-dimensional PCA scores.

print("Z shape:", Z1_w.shape)
print("1D scores:", np.round(Z1_w.ravel(), 3))

assert Z1_w.shape == (4, 1)

▶ What you'll see: four points have become four one-dimensional scores ordered along the main diagonal.

In [ ]:
plt.figure(figsize=(4.6, 2.6))
plt.scatter(Z1_w[:, 0], np.zeros_like(Z1_w[:, 0]), s=90, color="purple")
for i_w, z_w in enumerate(Z1_w[:, 0]):
    plt.text(z_w, 0.03, f"x{i_w}", ha="center")
plt.axhline(0, color="gray", linewidth=0.8)
plt.yticks([]); plt.xlabel("PC1 score"); plt.title("5: 2D points compressed to 1D"); plt.show()

▶ What you'll see: the points lie on a single line; PCA preserved their positions along the largest-spread direction.

*Why it's done this way:* The projection $X_cV_r$ is a dot product with orthonormal directions. Orthogonality prevents duplicate information between components, and truncating to $r$ columns keeps the coordinates that carry the most variance under a rank-$r$ linear bottleneck.

### 6. Explained variance turns singular values into a rank choice

The singular values produce energies $\sigma_j^2$. Dividing each energy by the total tells us how much variance each component explains. For the tiny matrix from the lesson content, the first component explains $8/(8+2)=0.8$ of the variance.

In [ ]:
energy_w = s_w ** 2  # component energies before dividing by m-1; ratios are unchanged.
ratio_w = energy_w / energy_w.sum()
cumulative_w = np.cumsum(ratio_w)

print("energies:", np.round(energy_w, 3))
print("explained ratios:", np.round(ratio_w, 3))
print("cumulative:", np.round(cumulative_w, 3))

assert round(float(ratio_w[0]), 3) == 0.8

▶ What you'll see: PC1 explains 80% of the variance and both components together explain 100%.

In [ ]:
plt.figure(figsize=(4.2, 3))
plt.bar(["PC1", "PC2"], ratio_w, color=["teal", "orange"])
plt.plot(["PC1", "PC2"], cumulative_w, marker="o", color="crimson", label="cumulative")
plt.ylim(0, 1.05); plt.ylabel("fraction of variance")
plt.title("6: explained variance ratio"); plt.legend(); plt.show()

▶ What you'll see: the first bar is 0.8, and the cumulative line reaches 1.0 after PC2.

*Why it's done this way:* Rank selection is a modeling decision. A one-component PCA keeps the dominant 80% pattern and discards the 20% orthogonal detail; that can denoise data, but it can also erase a weaker signal if that signal matters for the application.

### 7. Reconstruction shows exactly what PCA discards

A PCA reconstruction maps compressed scores back to the original feature space: $\hat X=Z_rV_r^\top+\mu$. Comparing $X$ and $\hat X$ makes the approximation concrete. The lost information is the component we chose not to keep.

In [ ]:
Xhat1_w = Z1_w @ V1_w.T + mean_w  # reconstruct from only PC1 and add the mean back.
errors_w = np.linalg.norm(X_w - Xhat1_w, axis=1)  # per-point reconstruction distance.

print("rank-1 reconstruction:\n", np.round(Xhat1_w, 3))
print("per-point errors:", np.round(errors_w, 3))

assert round(float(np.linalg.norm(X_w - Xhat1_w)), 3) == 1.414

▶ What you'll see: reconstructed points sit on the main diagonal, and the total rank-1 error is about 1.414.

In [ ]:
plt.figure(figsize=(4.5, 3.8))
plt.scatter(X_w[:, 0], X_w[:, 1], s=80, color="steelblue", label="original")
plt.scatter(Xhat1_w[:, 0], Xhat1_w[:, 1], s=80, color="darkorange", label="rank-1 reconstruction")
for i_w in range(X_w.shape[0]):
    plt.plot([X_w[i_w, 0], Xhat1_w[i_w, 0]], [X_w[i_w, 1], Xhat1_w[i_w, 1]], color="gray", linestyle="--")
plt.axis("equal"); plt.title("7: reconstruction error is discarded PC2 detail")
plt.legend(); plt.show()

▶ What you'll see: dashed lines connect each original point to its projection on the PC1 line.

*Why it's done this way:* PCA's reconstruction is the best least-squares rank-$r$ linear approximation when directions are chosen by SVD. Looking at the residuals is how we audit whether compression removed noise, harmless detail, or an important pattern.

### 8. Scaling decides what PCA is allowed to see

PCA is sensitive to feature scale because covariance uses squared units. If one feature is measured in much larger units, its variance can dominate the first component even if the feature is not conceptually more important. Standardizing to unit variance changes the question from "largest raw-unit variance" to "largest correlation-pattern variance."

In [ ]:
X_scale_w = np.column_stack([np.array([1., 2., 3., 4., 5.]),
                             100 * np.array([1.0, 1.1, 0.9, 1.2, 0.8])])  # second feature has large units.
X_scale_c_w = X_scale_w - X_scale_w.mean(axis=0)
std_w = X_scale_w.std(axis=0, ddof=1)
X_std_w = X_scale_c_w / std_w

print("feature stds:", np.round(std_w, 3))

assert std_w[1] >= 10 * std_w[0]

▶ What you'll see: feature 1's standard deviation is much larger because of units.

In [ ]:
_, _, Vt_raw_w = np.linalg.svd(X_scale_c_w, full_matrices=False)
_, _, Vt_std_w = np.linalg.svd(X_std_w, full_matrices=False)

print("raw PC1 direction:", np.round(Vt_raw_w[0], 3))
print("standardized PC1 direction:", np.round(Vt_std_w[0], 3))

▶ What you'll see: raw PCA points mostly along the large-unit feature, while standardized PCA balances the coordinates.

In [ ]:
plt.figure(figsize=(4.5, 3.2))
plt.bar(["raw |feature0|", "raw |feature1|"], np.abs(Vt_raw_w[0]), color="crimson", alpha=0.75)
plt.bar(["std |feature0|", "std |feature1|"], np.abs(Vt_std_w[0]), color="seagreen", alpha=0.75)
plt.xticks(rotation=20); plt.ylabel("absolute PC1 loading")
plt.title("8: scaling changes the PCA lens"); plt.show()

▶ What you'll see: the dominant loading changes after standardization, proving preprocessing is part of the model.

*Why it's done this way:* PCA optimizes variance exactly as represented numerically. Scaling is therefore not cosmetic; it defines the metric of importance. If raw units are meaningful, raw PCA may be right. If features should have equal starting influence, standardize first.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per mechanic in this lesson. Each uses a handful of small
> numbers, prints every intermediate value with an inline `# ->` showing the result, draws one
> picture, and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · Centering subtracts the feature means

PCA studies spread around the average point. Centering turns each row into a deviation from the
feature means.

In [ ]:
import numpy as np                              # arrays + centering.
import matplotlib.pyplot as plt                 # one picture per toy.

t1_rng = np.random.default_rng(0)               # seeded generator for reproducible toys.
t1_X = np.array([[1.0, 1.0], [2.0, 1.0], [3.0, 2.0], [4.0, 4.0], [5.0, 4.0], [6.0, 5.0]])

print("raw data:", t1_X.tolist())               # -> [[1.0, 1.0], [2.0, 1.0], [3.0, 2.0], [4.0, 4.0], [5.0, 4.0], [6.0, 5.0]]

t1_mean = t1_X.mean(axis=0)

print("feature means:", np.round(t1_mean, 3).tolist()) # -> [3.5, 2.833]

t1_Xc = t1_X - t1_mean

print("centered data:", np.round(t1_Xc, 3).tolist()) # -> [[-2.5, -1.833], [-1.5, -1.833], [-0.5, -0.833], [0.5, 1.167], [1.5, 1.167], [2.5, 2.167]]

t1_centered_mean = t1_Xc.mean(axis=0)

print("centered means:", np.round(t1_centered_mean, 6).tolist()) # -> [0.0, -0.0]

assert np.allclose(t1_centered_mean, [0.0, 0.0])

plt.figure(figsize=(4.6, 3.3))
plt.scatter(t1_X[:, 0], t1_X[:, 1], s=80, color="steelblue", label="raw")
plt.scatter(t1_Xc[:, 0], t1_Xc[:, 1], s=80, color="darkorange", label="centered")
plt.axhline(0, color="gray", linewidth=0.8)
plt.axvline(0, color="gray", linewidth=0.8)
plt.title("Toy 1 · centering moves the cloud")
plt.xlabel("feature 0")
plt.ylabel("feature 1")
plt.legend()
plt.show()

▶ What you'll see: the centered orange cloud has the same shape but mean zero.

### ✍️ Toy 2 · Covariance summarizes spread and co-movement

After centering, covariance is the scaled dot-product table `Xc.T @ Xc / (m-1)`.

In [ ]:
import numpy as np                              # arrays + covariance.

t2_rng = np.random.default_rng(0)               # seeded generator for reproducible toys.
t2_X = np.array([[1.0, 1.0], [2.0, 1.0], [3.0, 2.0], [4.0, 4.0], [5.0, 4.0], [6.0, 5.0]])

print("raw data:", t2_X.tolist())               # -> [[1.0, 1.0], [2.0, 1.0], [3.0, 2.0], [4.0, 4.0], [5.0, 4.0], [6.0, 5.0]]

t2_mean = t2_X.mean(axis=0)

print("feature means:", np.round(t2_mean, 3).tolist()) # -> [3.5, 2.833]

t2_Xc = t2_X - t2_mean

print("centered data:", np.round(t2_Xc, 3).tolist()) # -> [[-2.5, -1.833], [-1.5, -1.833], [-0.5, -0.833], [0.5, 1.167], [1.5, 1.167], [2.5, 2.167]]

t2_gram = t2_Xc.T @ t2_Xc

print("Xc.T @ Xc:", np.round(t2_gram, 3).tolist()) # -> [[17.5, 15.5], [15.5, 14.833]]

t2_m = t2_X.shape[0]

print("sample size:", t2_m)                     # -> 6

t2_C = t2_gram / (t2_m - 1)

print("covariance:", np.round(t2_C, 3).tolist()) # -> [[3.5, 3.1], [3.1, 2.967]]

assert np.allclose(np.round(t2_C, 3), [[3.5, 3.1], [3.1, 2.967]])

plt.figure(figsize=(3.8, 3.2))
plt.imshow(t2_C, cmap="coolwarm", vmin=-4, vmax=4)
plt.colorbar(label="covariance")
plt.xticks([0, 1], ["f0", "f1"])
plt.yticks([0, 1], ["f0", "f1"])
plt.title("Toy 2 · covariance matrix")
plt.show()

▶ What you'll see: positive diagonal and off-diagonal entries, so both features spread and rise together.

### ✍️ Toy 3 · Eigenvectors point along principal directions

The eigenvectors of the covariance matrix are PCA directions, and eigenvalues say how much variance
each direction captures.

In [ ]:
import numpy as np                              # arrays + eigendecomposition.

t3_rng = np.random.default_rng(0)               # seeded generator for reproducible toys.
t3_X = np.array([[1.0, 1.0], [2.0, 1.0], [3.0, 2.0], [4.0, 4.0], [5.0, 4.0], [6.0, 5.0]])

print("raw data:", t3_X.tolist())               # -> [[1.0, 1.0], [2.0, 1.0], [3.0, 2.0], [4.0, 4.0], [5.0, 4.0], [6.0, 5.0]]

t3_Xc = t3_X - t3_X.mean(axis=0)

print("centered data:", np.round(t3_Xc, 3).tolist()) # -> [[-2.5, -1.833], [-1.5, -1.833], [-0.5, -0.833], [0.5, 1.167], [1.5, 1.167], [2.5, 2.167]]

t3_C = (t3_Xc.T @ t3_Xc) / (t3_X.shape[0] - 1)

print("covariance:", np.round(t3_C, 3).tolist()) # -> [[3.5, 3.1], [3.1, 2.967]]

t3_evals, t3_evecs = np.linalg.eigh(t3_C)

print("unsorted eigenvalues:", np.round(t3_evals, 3).tolist()) # -> [0.122, 6.345]

t3_order = np.argsort(t3_evals)[::-1]

print("descending order:", t3_order.tolist())   # -> [1, 0]

t3_evals = t3_evals[t3_order]

print("sorted eigenvalues:", np.round(t3_evals, 3).tolist()) # -> [6.345, 0.122]

t3_evecs = t3_evecs[:, t3_order]

print("sorted eigenvectors:", np.round(t3_evecs, 3).tolist()) # -> [[-0.737, 0.676], [-0.676, -0.737]]

assert np.allclose(np.round(t3_evals, 3), [6.345, 0.122])

plt.figure(figsize=(4.4, 3.6))
plt.scatter(t3_Xc[:, 0], t3_Xc[:, 1], s=80, color="steelblue")
for t3_j, t3_color in enumerate(["crimson", "seagreen"]):
    t3_vec = t3_evecs[:, t3_j] * np.sqrt(t3_evals[t3_j])
    plt.arrow(0, 0, t3_vec[0], t3_vec[1], head_width=0.08, color=t3_color, length_includes_head=True)
    plt.text(t3_vec[0] * 1.08, t3_vec[1] * 1.08, f"PC{t3_j + 1}", color=t3_color)
plt.axhline(0, color="gray", linewidth=0.8)
plt.axvline(0, color="gray", linewidth=0.8)
plt.axis("equal")
plt.title("Toy 3 · covariance eigenvectors")
plt.show()

▶ What you'll see: PC1 follows the long direction of the centered cloud; PC2 is perpendicular.

### ✍️ Toy 4 · SVD returns the same PCA directions

SVD works directly on the centered matrix. Its right singular vectors match covariance eigenvectors
up to harmless sign flips.

In [ ]:
import numpy as np                              # arrays + SVD.

t4_rng = np.random.default_rng(0)               # seeded generator for reproducible toys.
t4_X = np.array([[1.0, 1.0], [2.0, 1.0], [3.0, 2.0], [4.0, 4.0], [5.0, 4.0], [6.0, 5.0]])

print("raw data:", t4_X.tolist())               # -> [[1.0, 1.0], [2.0, 1.0], [3.0, 2.0], [4.0, 4.0], [5.0, 4.0], [6.0, 5.0]]

t4_Xc = t4_X - t4_X.mean(axis=0)

print("centered data:", np.round(t4_Xc, 3).tolist()) # -> [[-2.5, -1.833], [-1.5, -1.833], [-0.5, -0.833], [0.5, 1.167], [1.5, 1.167], [2.5, 2.167]]

t4_U, t4_s, t4_Vt = np.linalg.svd(t4_Xc, full_matrices=False)

print("singular values:", np.round(t4_s, 3).tolist()) # -> [5.632, 0.781]
print("Vt:", np.round(t4_Vt, 3).tolist())       # -> [[0.737, 0.676], [-0.676, 0.737]]

t4_variance = (t4_s ** 2) / (t4_X.shape[0] - 1)

print("variance from SVD:", np.round(t4_variance, 3).tolist()) # -> [6.345, 0.122]

t4_C = (t4_Xc.T @ t4_Xc) / (t4_X.shape[0] - 1)
t4_evals, t4_evecs = np.linalg.eigh(t4_C)
t4_order = np.argsort(t4_evals)[::-1]
t4_evecs = t4_evecs[:, t4_order]
t4_alignment = np.abs(np.sum(t4_Vt.T * t4_evecs, axis=0))

print("absolute alignments:", np.round(t4_alignment, 3).tolist()) # -> [1.0, 1.0]

assert np.allclose(np.round(t4_alignment, 3), [1.0, 1.0])

plt.figure(figsize=(4.2, 3.0))
plt.bar(["PC1", "PC2"], t4_s, color=["teal", "orange"])
plt.title("Toy 4 · singular values from centered data")
plt.ylabel("singular value")
plt.show()

▶ What you'll see: SVD and eigendecomposition agree on directions and variance, except for arbitrary signs.

### ✍️ Toy 5 · Projection creates one-dimensional scores

Keeping PC1 turns each two-dimensional centered point into one coordinate along the highest-variance
axis.

In [ ]:
import numpy as np                              # arrays + projection.

t5_rng = np.random.default_rng(0)               # seeded generator for reproducible toys.
t5_X = np.array([[1.0, 1.0], [2.0, 1.0], [3.0, 2.0], [4.0, 4.0], [5.0, 4.0], [6.0, 5.0]])

print("raw data:", t5_X.tolist())               # -> [[1.0, 1.0], [2.0, 1.0], [3.0, 2.0], [4.0, 4.0], [5.0, 4.0], [6.0, 5.0]]

t5_mean = t5_X.mean(axis=0)

print("feature means:", np.round(t5_mean, 3).tolist()) # -> [3.5, 2.833]

t5_Xc = t5_X - t5_mean

print("centered data:", np.round(t5_Xc, 3).tolist()) # -> [[-2.5, -1.833], [-1.5, -1.833], [-0.5, -0.833], [0.5, 1.167], [1.5, 1.167], [2.5, 2.167]]

t5_U, t5_s, t5_Vt = np.linalg.svd(t5_Xc, full_matrices=False)

print("Vt:", np.round(t5_Vt, 3).tolist())       # -> [[0.737, 0.676], [-0.676, 0.737]]

t5_V1 = t5_Vt.T[:, :1]

print("PC1 direction:", np.round(t5_V1.ravel(), 3).tolist()) # -> [0.737, 0.676]

t5_scores = t5_Xc @ t5_V1

print("PC1 scores:", np.round(t5_scores.ravel(), 3).tolist()) # -> [-3.082, -2.345, -0.932, 1.157, 1.894, 3.307]

assert t5_scores.shape == (6, 1)

plt.figure(figsize=(4.8, 2.5))
plt.scatter(t5_scores[:, 0], np.zeros_like(t5_scores[:, 0]), s=90, color="purple")
for t5_i, t5_z in enumerate(t5_scores[:, 0]):
    plt.text(t5_z, 0.03, f"x{t5_i}", ha="center")
plt.axhline(0, color="gray", linewidth=0.8)
plt.yticks([])
plt.title("Toy 5 · 2D points compressed to PC1")
plt.xlabel("PC1 score")
plt.show()

▶ What you'll see: six 2D rows become six ordered 1D scores along the main direction.

### ✍️ Toy 6 · Explained variance turns energy into a rank choice

Squared singular values are component energy. Dividing by total energy gives the explained variance
fraction.

In [ ]:
import numpy as np                              # arrays + explained-variance ratios.

t6_rng = np.random.default_rng(0)               # seeded generator for reproducible toys.
t6_X = np.array([[1.0, 1.0], [2.0, 1.0], [3.0, 2.0], [4.0, 4.0], [5.0, 4.0], [6.0, 5.0]])

print("raw data:", t6_X.tolist())               # -> [[1.0, 1.0], [2.0, 1.0], [3.0, 2.0], [4.0, 4.0], [5.0, 4.0], [6.0, 5.0]]

t6_Xc = t6_X - t6_X.mean(axis=0)

print("centered data:", np.round(t6_Xc, 3).tolist()) # -> [[-2.5, -1.833], [-1.5, -1.833], [-0.5, -0.833], [0.5, 1.167], [1.5, 1.167], [2.5, 2.167]]

t6_s = np.linalg.svd(t6_Xc, full_matrices=False)[1]

print("singular values:", np.round(t6_s, 3).tolist()) # -> [5.632, 0.781]

t6_energy = t6_s ** 2

print("component energies:", np.round(t6_energy, 3).tolist()) # -> [31.724, 0.609]

t6_ratio = t6_energy / t6_energy.sum()

print("explained ratios:", np.round(t6_ratio, 3).tolist()) # -> [0.981, 0.019]

t6_cumulative = np.cumsum(t6_ratio)

print("cumulative ratios:", np.round(t6_cumulative, 3).tolist()) # -> [0.981, 1.0]

assert round(float(t6_ratio[0]), 3) == 0.981

plt.figure(figsize=(4.2, 3.0))
plt.bar(["PC1", "PC2"], t6_ratio, color=["teal", "orange"])
plt.plot(["PC1", "PC2"], t6_cumulative, marker="o", color="black", label="cumulative")
plt.ylim(0, 1.05)
plt.title("Toy 6 · most energy is PC1")
plt.ylabel("fraction")
plt.legend()
plt.show()

▶ What you'll see: PC1 explains about 98.1% of this tiny cloud's variance.

### ✍️ Toy 7 · Reconstruction exposes the discarded component

Project to PC1, map back to feature space, and measure how far each reconstructed point sits from
its original row.

In [ ]:
import numpy as np                              # arrays + PCA reconstruction.

t7_rng = np.random.default_rng(0)               # seeded generator for reproducible toys.
t7_X = np.array([[1.0, 1.0], [2.0, 1.0], [3.0, 2.0], [4.0, 4.0], [5.0, 4.0], [6.0, 5.0]])

print("raw data:", t7_X.tolist())               # -> [[1.0, 1.0], [2.0, 1.0], [3.0, 2.0], [4.0, 4.0], [5.0, 4.0], [6.0, 5.0]]

t7_mean = t7_X.mean(axis=0)

print("feature means:", np.round(t7_mean, 3).tolist()) # -> [3.5, 2.833]

t7_Xc = t7_X - t7_mean
t7_U, t7_s, t7_Vt = np.linalg.svd(t7_Xc, full_matrices=False)

print("singular values:", np.round(t7_s, 3).tolist()) # -> [5.632, 0.781]

t7_V1 = t7_Vt.T[:, :1]

print("PC1 direction:", np.round(t7_V1.ravel(), 3).tolist()) # -> [0.737, 0.676]

t7_scores = t7_Xc @ t7_V1

print("PC1 scores:", np.round(t7_scores.ravel(), 3).tolist()) # -> [-3.082, -2.345, -0.932, 1.157, 1.894, 3.307]

t7_Xhat = t7_scores @ t7_V1.T + t7_mean

print("rank-1 reconstruction:", np.round(t7_Xhat, 3).tolist()) # -> [[1.23, 0.75], [1.772, 1.248], [2.813, 2.203], [4.353, 3.616], [4.895, 4.114], [5.936, 5.069]]

t7_errors = np.linalg.norm(t7_X - t7_Xhat, axis=1)

print("per-point errors:", np.round(t7_errors, 3).tolist()) # -> [0.34, 0.337, 0.276, 0.522, 0.155, 0.094]

t7_total_error = float(np.linalg.norm(t7_X - t7_Xhat))

print("total error:", round(t7_total_error, 3))  # -> 0.781

assert round(t7_total_error, 3) == 0.781

plt.figure(figsize=(4.5, 3.5))
plt.scatter(t7_X[:, 0], t7_X[:, 1], s=80, color="steelblue", label="original")
plt.scatter(t7_Xhat[:, 0], t7_Xhat[:, 1], s=80, color="darkorange", label="rank-1")
for t7_i in range(len(t7_X)):
    plt.plot([t7_X[t7_i, 0], t7_Xhat[t7_i, 0]], [t7_X[t7_i, 1], t7_Xhat[t7_i, 1]], color="gray", linestyle="--")
plt.axis("equal")
plt.title("Toy 7 · residuals are discarded PC2")
plt.legend()
plt.show()

▶ What you'll see: reconstructed points lie on the PC1 line, with dashed residuals to the originals.

### ✍️ Toy 8 · Scaling changes the PCA lens

Raw PCA follows the largest numeric variance. Standardizing changes the question so each feature
starts with equal variance.

In [ ]:
import numpy as np                              # arrays + standardized PCA.

t8_rng = np.random.default_rng(0)               # seeded generator for reproducible toys.
t8_X = np.column_stack([np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0]), np.array([10.0, 30.0, 20.0, 50.0, 40.0, 60.0])])

print("raw data:", t8_X.tolist())               # -> [[1.0, 10.0], [2.0, 30.0], [3.0, 20.0], [4.0, 50.0], [5.0, 40.0], [6.0, 60.0]]

t8_mean = t8_X.mean(axis=0)

print("feature means:", np.round(t8_mean, 3).tolist()) # -> [3.5, 35.0]

t8_Xc = t8_X - t8_mean
t8_std = t8_X.std(axis=0, ddof=1)

print("sample stds:", np.round(t8_std, 3).tolist()) # -> [1.871, 18.708]

t8_Z = t8_Xc / t8_std

print("standardized stds:", np.round(t8_Z.std(axis=0, ddof=1), 3).tolist()) # -> [1.0, 1.0]

t8_raw_Vt = np.linalg.svd(t8_Xc, full_matrices=False)[2]

print("raw PC1 direction:", np.round(t8_raw_Vt[0], 3).tolist()) # -> [-0.088, -0.996]

t8_std_Vt = np.linalg.svd(t8_Z, full_matrices=False)[2]

print("standardized PC1 direction:", np.round(t8_std_Vt[0], 3).tolist()) # -> [0.707, 0.707]

t8_raw_abs = np.abs(t8_raw_Vt[0])

print("raw |loadings|:", np.round(t8_raw_abs, 3).tolist()) # -> [0.088, 0.996]

t8_std_abs = np.abs(t8_std_Vt[0])

print("standardized |loadings|:", np.round(t8_std_abs, 3).tolist()) # -> [0.707, 0.707]

assert t8_raw_abs[1] > 10 * t8_raw_abs[0]

plt.figure(figsize=(4.8, 3.0))
plt.bar(["raw f0", "raw f1", "std f0", "std f1"], [t8_raw_abs[0], t8_raw_abs[1], t8_std_abs[0], t8_std_abs[1]], color=["crimson", "crimson", "teal", "teal"])
plt.title("Toy 8 · PC1 loadings before/after scaling")
plt.ylabel("absolute loading")
plt.xticks(rotation=15)
plt.show()

▶ What you'll see: raw PC1 is almost entirely feature 1, while standardized PC1 balances both features.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, centering, covariance, eigendecomposition, SVD, and assertions.
import matplotlib.pyplot as plt  # load Matplotlib for every scatter, bar chart, heatmap, and diagnostic line plot.
np.random.seed(0)  # make stochastic examples reproducible.

def pca_fit(X, r=None):  # fit PCA from scratch with centering and SVD.
    X = np.asarray(X, dtype=float)  # convert input to a numeric array.
    mean = X.mean(axis=0)  # compute one mean per feature.
    Xc = X - mean  # center so PCA measures variation around the cloud average.
    U, s, Vt = np.linalg.svd(Xc, full_matrices=False)  # factor centered data without sklearn.
    V = Vt.T  # store principal directions as columns.
    if r is not None:  # optionally keep only the requested rank.
        V = V[:, :r]  # truncate directions.
        s = s[:r]  # truncate singular values consistently.
    return mean, V, s, Xc  # return the pieces used by all examples.

def pca_transform(X, mean, V):  # project data into PCA score coordinates.
    return (np.asarray(X, dtype=float) - mean) @ V  # centered dot products with principal directions.

def pca_reconstruct(Z, mean, V):  # map PCA scores back to feature space.
    return Z @ V.T + mean  # undo projection and add the mean baseline back.

def explained_ratio(s):  # convert singular values to variance fractions.
    energy = np.asarray(s, dtype=float) ** 2  # component energy is squared singular value.
    return energy / energy.sum()  # normalize energies into fractions that sum to one.

## 🟢 Basics (warm-up)

### Basic 1 — Center a tiny data matrix

**Goal.** Subtract feature means, because PCA works with deviations from the average point rather than raw coordinates. We build it in 2 steps.

In [ ]:
X_b1 = np.array([[1., 2.], [2., 1.], [3., 4.], [4., 3.]])  # define the lesson's tiny 4x2 matrix.
mean_b1 = X_b1.mean(axis=0)  # compute column means.

print("means:", mean_b1)  # inspect the baseline that will be removed.

assert np.allclose(mean_b1, [2.5, 2.5])  # verify the lesson arithmetic.

▶ What you'll see: each feature has mean 2.5.

In [ ]:
Xc_b1 = X_b1 - mean_b1  # center each feature by subtracting its own mean.

print("centered:\n", Xc_b1)  # inspect deviations from the mean.
print("centered column means:", Xc_b1.mean(axis=0))  # confirm centering worked.

plt.figure(figsize=(4, 3))
plt.scatter(X_b1[:, 0], X_b1[:, 1], color="steelblue", label="raw")
plt.scatter(Xc_b1[:, 0], Xc_b1[:, 1], color="orange", label="centered")
plt.axhline(0, color="gray", linewidth=0.8); plt.axvline(0, color="gray", linewidth=0.8)
plt.title("Basic 1: raw vs centered points"); plt.legend(); plt.show()

▶ What you'll see: centering preserves shape but moves the cloud to mean zero.

👀 Takeaway: centering removes location so PCA can focus on spread.

### Basic 2 — Compute a covariance matrix

**Goal.** Summarize spread with covariance, because PCA directions are chosen from feature variances and co-movements. We build it in 2 steps.

In [ ]:
X_b2 = np.array([[1., 2.], [2., 1.], [3., 4.], [4., 3.]])  # recreate the toy data locally.
Xc_b2 = X_b2 - X_b2.mean(axis=0)  # center before covariance.
gram_b2 = Xc_b2.T @ Xc_b2  # accumulate centered cross-products.

print("centered cross-products:\n", gram_b2)  # inspect the numerator of covariance.

▶ What you'll see: the cross-product matrix has positive diagonal and off-diagonal entries.

In [ ]:
C_b2 = gram_b2 / (X_b2.shape[0] - 1)  # divide by m-1 for sample covariance.

print("covariance:\n", np.round(C_b2, 3))  # inspect the spread summary.

assert np.allclose(np.round(C_b2, 3), [[1.667, 1.000], [1.000, 1.667]])  # verify the numeric result.
plt.figure(figsize=(4, 3))
plt.imshow(C_b2, cmap="coolwarm", vmin=-2, vmax=2)
plt.colorbar(label="covariance")
plt.title("Basic 2: covariance matrix"); plt.show()

▶ What you'll see: the positive off-diagonal shows that the two features tend to increase together.

👀 Takeaway: covariance is the compact object PCA diagonalizes.

### Basic 3 — Find principal directions with eigenvectors

**Goal.** Extract eigenvectors of covariance, because they are the orthogonal axes of maximum variance. We build it in 3 steps.

In [ ]:
C_b3 = np.array([[1.6666667, 1.0], [1.0, 1.6666667]])  # use the covariance from Basic 2.
evals_b3, evecs_b3 = np.linalg.eigh(C_b3)  # compute eigenpairs for the symmetric matrix.

print("unsorted eigenvalues:", np.round(evals_b3, 3))  # inspect the raw order returned by NumPy.

▶ What you'll see: the eigenvalues are present but not in PCA's descending-variance order.

In [ ]:
order_b3 = np.argsort(evals_b3)[::-1]  # sort largest variance first.
evals_b3 = evals_b3[order_b3]  # reorder eigenvalues.
evecs_b3 = evecs_b3[:, order_b3]  # reorder directions consistently.

print("sorted eigenvalues:", np.round(evals_b3, 3))  # inspect component variances.
print("PC directions:\n", np.round(evecs_b3, 3))  # inspect principal axes.

assert np.allclose(np.round(evals_b3, 3), [2.667, 0.667])

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["PC1", "PC2"], evals_b3, color=["teal", "orange"])
plt.title("Basic 3: eigenvalue = variance on that PC")
plt.ylabel("variance"); plt.show()

▶ What you'll see: PC1 has much larger variance than PC2.

👀 Takeaway: PCA sorts eigenvectors by how much variance their eigenvalues capture.

### Basic 4 — Project points onto PC1

**Goal.** Compute one-dimensional PCA scores, because dimensionality reduction is projection onto the kept directions. We build it in 2 steps.

In [ ]:
X_b4 = np.array([[1., 2.], [2., 1.], [3., 4.], [4., 3.]])  # recreate the toy points.
mean_b4, V_b4, s_b4, Xc_b4 = pca_fit(X_b4, r=1)  # fit PCA and keep only PC1.

print("PC1 direction:", np.round(V_b4[:, 0], 3))  # inspect the kept direction.

▶ What you'll see: PC1 points along the diagonal direction of the cloud, up to sign.

In [ ]:
Z_b4 = pca_transform(X_b4, mean_b4, V_b4)  # project centered points onto PC1.

print("PC1 scores:", np.round(Z_b4.ravel(), 3))  # inspect the 1D representation.

assert Z_b4.shape == (4, 1)  # verify the reduced shape.
plt.figure(figsize=(4.5, 2.5))
plt.scatter(Z_b4[:, 0], np.zeros(4), s=80, color="purple")
plt.yticks([]); plt.xlabel("PC1 score"); plt.title("Basic 4: projected 1D scores"); plt.show()

▶ What you'll see: each 2D point is now one signed coordinate along PC1.

👀 Takeaway: PCA scores are centered dot products with principal directions.

### Basic 5 — Read singular values as component energy

**Goal.** Convert SVD singular values into explained variance ratios, because rank choice depends on how much energy each component keeps. We build it in 2 steps.

In [ ]:
X_b5 = np.array([[1., 2.], [2., 1.], [3., 4.], [4., 3.]])  # define the tiny matrix.
mean_b5, V_b5, s_b5, Xc_b5 = pca_fit(X_b5)  # fit full PCA by SVD.
energy_b5 = s_b5 ** 2  # square singular values to get component energies.

print("singular values:", np.round(s_b5, 3))  # inspect SVD scale.
print("energies:", np.round(energy_b5, 3))  # inspect variance numerators.

assert np.allclose(np.round(s_b5, 3), [2.828, 1.414])

▶ What you'll see: singular values produce energies 8 and 2.

In [ ]:
ratio_b5 = explained_ratio(s_b5)  # normalize energy into fractions.

print("explained variance ratio:", np.round(ratio_b5, 3))  # inspect retained fraction per PC.

assert round(float(ratio_b5[0]), 3) == 0.8  # verify the lesson's 80% number.
plt.figure(figsize=(4, 3))
plt.bar(["PC1", "PC2"], ratio_b5, color="teal")
plt.ylim(0, 1); plt.title("Basic 5: explained variance"); plt.show()

▶ What you'll see: PC1 keeps 80% of the total variance.

👀 Takeaway: explained variance ratio tells how much structure each component preserves.

### Basic 6 — Reconstruct from one component

**Goal.** Map compressed scores back to the original space, because reconstruction reveals what PCA compression discarded. We build it in 3 steps.

In [ ]:
X_b6 = np.array([[1., 2.], [2., 1.], [3., 4.], [4., 3.]])  # define the toy points.
mean_b6, V_b6, s_b6, Xc_b6 = pca_fit(X_b6, r=1)  # keep only the first principal direction.
Z_b6 = pca_transform(X_b6, mean_b6, V_b6)  # compute rank-1 scores.

print("Z shape:", Z_b6.shape)  # inspect compressed shape.

▶ What you'll see: four rows now have one PCA coordinate each.

In [ ]:
Xhat_b6 = pca_reconstruct(Z_b6, mean_b6, V_b6)  # reconstruct from the rank-1 representation.

print("rank-1 reconstruction:\n", np.round(Xhat_b6, 3))  # inspect approximate points.

err_b6 = float(np.linalg.norm(X_b6 - Xhat_b6))  # compute total reconstruction error.

print("total reconstruction error:", round(err_b6, 3))  # inspect lost information.

assert round(err_b6, 3) == 1.414

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(X_b6[:, 0], X_b6[:, 1], color="steelblue", label="original")
plt.scatter(Xhat_b6[:, 0], Xhat_b6[:, 1], color="orange", label="rank-1")
plt.axis("equal"); plt.title("Basic 6: reconstruction from PC1"); plt.legend(); plt.show()

▶ What you'll see: reconstructed points lie on the PC1 line rather than exactly on the originals.

👀 Takeaway: PCA compression trades lower dimension for reconstruction error.

### Basic 7 — Check orthogonality of components

**Goal.** Verify principal directions are perpendicular, because PCA components should not duplicate the same variance. We build it in 2 steps.

In [ ]:
X_b7 = np.array([[1., 2.], [2., 1.], [3., 4.], [4., 3.]])  # recreate the toy data.
mean_b7, V_b7, s_b7, Xc_b7 = pca_fit(X_b7)  # fit full PCA.
dot_b7 = float(V_b7[:, 0] @ V_b7[:, 1])  # compute dot product between PC directions.

print("PC dot product:", round(dot_b7, 6))  # inspect near-zero orthogonality.

assert abs(dot_b7) < 1e-12

▶ What you'll see: the dot product is essentially zero.

In [ ]:
I_b7 = V_b7.T @ V_b7  # orthonormal direction matrix should multiply to identity.

print("V.T @ V:\n", np.round(I_b7, 3))  # inspect orthonormality.

plt.figure(figsize=(4, 3))
plt.imshow(I_b7, cmap="Greys", vmin=0, vmax=1)
plt.colorbar(label="value"); plt.title("Basic 7: principal directions are orthonormal"); plt.show()

▶ What you'll see: the identity matrix, with ones on the diagonal and zeros off the diagonal.

👀 Takeaway: orthogonal components separate variance into non-overlapping axes.

### Basic 8 — Compare raw and centered projection

**Goal.** Show why the mean must be subtracted before projection, because projecting raw coordinates mixes location with variation. We build it in 2 steps.

In [ ]:
X_b8 = np.array([[11., 12.], [12., 11.], [13., 14.], [14., 13.]])  # same shape as the toy cloud, shifted by 10.
mean_b8, V_b8, s_b8, Xc_b8 = pca_fit(X_b8, r=1)  # fit PCA with correct centering.
raw_scores_b8 = X_b8 @ V_b8  # intentionally wrong: project without subtracting the mean.
centered_scores_b8 = pca_transform(X_b8, mean_b8, V_b8)  # correct centered projection.

print("raw score mean:", round(float(raw_scores_b8.mean()), 3))
print("centered score mean:", round(float(centered_scores_b8.mean()), 3))

assert abs(float(centered_scores_b8.mean())) < 1e-12

▶ What you'll see: raw scores have a large offset, while centered scores average to zero.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["raw projection mean", "centered projection mean"], [float(raw_scores_b8.mean()), float(centered_scores_b8.mean())], color=["red", "green"])
plt.title("Basic 8: centering removes score offset")
plt.xticks(rotation=15); plt.show()

▶ What you'll see: the raw projection bar is large only because the cloud was shifted.

👀 Takeaway: PCA transform must use the training mean, not raw coordinates.

### Basic 9 — Inspect loadings as feature weights

**Goal.** Read a principal direction's coordinates, because loadings show how original features combine into a component. We build it in 2 steps.

In [ ]:
X_b9 = np.array([[1., 2.], [2., 1.], [3., 4.], [4., 3.]])  # define the tiny data.
mean_b9, V_b9, s_b9, Xc_b9 = pca_fit(X_b9)  # fit PCA.
loadings_b9 = V_b9[:, 0]  # PC1 feature weights.

print("PC1 loadings:", np.round(loadings_b9, 3))  # inspect weights for feature 0 and feature 1.

assert np.allclose(np.abs(np.round(loadings_b9, 3)), [0.707, 0.707])

▶ What you'll see: both features contribute equally in magnitude to PC1.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["feature 0", "feature 1"], loadings_b9, color="slateblue")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 9: PC1 loadings"); plt.ylabel("weight"); plt.show()

▶ What you'll see: equal-magnitude bars, meaning PC1 averages the two centered features up to sign.

👀 Takeaway: loadings explain which original features define a component.

### Basic 10 — Standardize before PCA when units differ

**Goal.** Put features on comparable scale, because raw covariance gives larger-unit features more influence. We build it in 3 steps.

In [ ]:
X_b10 = np.column_stack([np.array([1., 2., 3., 4., 5.]), np.array([100., 110., 90., 120., 80.])])  # second feature has larger units.
std_b10 = X_b10.std(axis=0, ddof=1)  # compute sample standard deviations.

print("standard deviations:", np.round(std_b10, 3))  # inspect scale mismatch.

assert std_b10[1] > std_b10[0]

▶ What you'll see: feature 1's spread is much larger in raw units.

In [ ]:
mean_raw_b10, V_raw_b10, s_raw_b10, Xc_raw_b10 = pca_fit(X_b10, r=1)  # fit raw PCA.
Xz_b10 = (X_b10 - X_b10.mean(axis=0)) / std_b10  # standardize each feature to unit sample variance.
mean_z_b10, V_z_b10, s_z_b10, Xc_z_b10 = pca_fit(Xz_b10, r=1)  # fit PCA after standardization.

print("raw PC1:", np.round(V_raw_b10[:, 0], 3))
print("standardized PC1:", np.round(V_z_b10[:, 0], 3))

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(["raw f0", "raw f1"], np.abs(V_raw_b10[:, 0]), color="crimson", alpha=0.7)
plt.bar(["std f0", "std f1"], np.abs(V_z_b10[:, 0]), color="seagreen", alpha=0.7)
plt.xticks(rotation=15); plt.ylabel("absolute loading")
plt.title("Basic 10: scaling changes loadings"); plt.show()

▶ What you'll see: raw PCA emphasizes the large-unit feature; standardized PCA changes the balance.

👀 Takeaway: preprocessing is part of the PCA model because it defines what variance means.

## 🟡 Easy

### Easy 1 — Build PCA with covariance eigendecomposition

**Goal.** Implement PCA through covariance and eigenvectors, because this is the direct mathematical definition of principal components. We build it in 3 steps.

In [ ]:
X_e1 = np.array([[1., 2.], [2., 1.], [3., 4.], [4., 3.]])  # define the lesson matrix.
mean_e1 = X_e1.mean(axis=0)  # compute feature means.
Xc_e1 = X_e1 - mean_e1  # center before covariance.
C_e1 = Xc_e1.T @ Xc_e1 / (X_e1.shape[0] - 1)  # sample covariance.

print("covariance:\n", np.round(C_e1, 3))

▶ What you'll see: the covariance matrix captures equal feature variances and positive co-movement.

In [ ]:
evals_e1, evecs_e1 = np.linalg.eigh(C_e1)  # diagonalize covariance.
idx_e1 = np.argsort(evals_e1)[::-1]  # sort components by variance.
evals_e1 = evals_e1[idx_e1]
evecs_e1 = evecs_e1[:, idx_e1]
Z_e1 = Xc_e1 @ evecs_e1  # project onto both components.

print("eigenvalues:", np.round(evals_e1, 3))
print("scores:\n", np.round(Z_e1, 3))

assert np.allclose(np.round(evals_e1, 3), [2.667, 0.667])

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(Z_e1[:, 0], Z_e1[:, 1], s=80, color="teal")
plt.axhline(0, color="gray", linewidth=0.8); plt.axvline(0, color="gray", linewidth=0.8)
plt.title("Easy 1: data in PCA coordinates")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.show()

▶ What you'll see: the transformed coordinates are aligned with the principal axes.

👀 Takeaway: covariance eigendecomposition gives PCA directions and PCA scores from scratch.

### Easy 2 — Match SVD PCA to covariance PCA

**Goal.** Verify that SVD on centered data gives the same principal directions as covariance PCA, because real implementations usually use SVD. We build it in 3 steps.

In [ ]:
X_e2 = np.array([[1., 2.], [2., 1.], [3., 4.], [4., 3.]])  # define the toy matrix.
Xc_e2 = X_e2 - X_e2.mean(axis=0)  # center the data.
U_e2, s_e2, Vt_e2 = np.linalg.svd(Xc_e2, full_matrices=False)  # compute SVD of centered data.

print("singular values:", np.round(s_e2, 3))

assert np.allclose(np.round(s_e2, 3), [2.828, 1.414])

▶ What you'll see: SVD exposes the same `[2.828, 1.414]` scale from the lesson content.

In [ ]:
C_e2 = Xc_e2.T @ Xc_e2 / (X_e2.shape[0] - 1)  # covariance for comparison.
evals_e2, evecs_e2 = np.linalg.eigh(C_e2)  # covariance eigenvectors.
evecs_e2 = evecs_e2[:, np.argsort(evals_e2)[::-1]]  # sort descending.
V_e2 = Vt_e2.T  # SVD directions.
alignment_e2 = np.abs(np.sum(V_e2 * evecs_e2, axis=0))  # compare up to sign.

print("direction alignment:", np.round(alignment_e2, 3))

assert np.allclose(np.round(alignment_e2, 3), [1.0, 1.0])

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["PC1 alignment", "PC2 alignment"], alignment_e2, color="purple")
plt.ylim(0, 1.05); plt.title("Easy 2: SVD and covariance directions agree"); plt.show()

▶ What you'll see: both alignment bars equal 1, allowing for sign flips.

👀 Takeaway: PCA can be computed through SVD without explicitly forming covariance.

### Easy 3 — Choose rank by cumulative explained variance

**Goal.** Select the smallest rank that reaches a variance threshold, because PCA compression needs an explicit retention rule. We build it in 3 steps.

In [ ]:
X_e3 = np.array([[1., 2.], [2., 1.], [3., 4.], [4., 3.]])  # define the tiny matrix.
mean_e3, V_e3, s_e3, Xc_e3 = pca_fit(X_e3)  # fit full PCA.
ratio_e3 = explained_ratio(s_e3)  # compute component fractions.
cum_e3 = np.cumsum(ratio_e3)  # cumulative retained variance.

print("ratios:", np.round(ratio_e3, 3), "cumulative:", np.round(cum_e3, 3))

▶ What you'll see: one component keeps 0.8, and two components keep 1.0.

In [ ]:
threshold_e3 = 0.90  # require 90% retained variance.
r_e3 = int(np.searchsorted(cum_e3, threshold_e3) + 1)  # first rank reaching the threshold.

print("rank for 90%:", r_e3)  # inspect selected rank.

assert r_e3 == 2  # 80% is not enough, so this toy needs both components for 90%.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot([1, 2], cum_e3, marker="o", color="teal")
plt.axhline(threshold_e3, color="crimson", linestyle="--", label="90% threshold")
plt.xticks([1, 2]); plt.ylim(0, 1.05)
plt.title("Easy 3: cumulative explained variance")
plt.xlabel("rank r"); plt.ylabel("cumulative fraction"); plt.legend(); plt.show()

▶ What you'll see: the 90% line is crossed only when rank reaches 2.

👀 Takeaway: explained-variance thresholds translate singular values into a concrete rank choice.

### Easy 4 — Compress and reconstruct a 3D cloud

**Goal.** Reduce correlated 3D data to 2D and reconstruct it, because PCA is most useful when several measured coordinates share a lower-dimensional structure. We build it in 4 steps.

In [ ]:
t_e4 = np.linspace(-2, 2, 30)  # create one underlying coordinate.
X_e4 = np.column_stack([t_e4, 2 * t_e4 + 0.2 * np.sin(3 * t_e4), -t_e4 + 0.1 * np.cos(4 * t_e4)])  # build three correlated features.
mean_e4, V_e4, s_e4, Xc_e4 = pca_fit(X_e4, r=2)  # fit rank-2 PCA.

print("X shape:", X_e4.shape, "V shape:", V_e4.shape)  # inspect input and component shapes.

▶ What you'll see: a 30×3 dataset is represented by two principal directions.

In [ ]:
Z_e4 = pca_transform(X_e4, mean_e4, V_e4)  # compress to 2D scores.
Xhat_e4 = pca_reconstruct(Z_e4, mean_e4, V_e4)  # reconstruct from the 2D bottleneck.
rmse_e4 = float(np.sqrt(np.mean((X_e4 - Xhat_e4) ** 2)))  # average reconstruction error.

print("Z shape:", Z_e4.shape, "reconstruction RMSE:", round(rmse_e4, 4))

assert Z_e4.shape == (30, 2)

In [ ]:
ratio_e4 = explained_ratio(s_e4)  # explained ratio among the kept components.

print("kept-component ratio:", np.round(ratio_e4, 3))  # inspect how the first two retained axes share energy.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.scatter(Z_e4[:, 0], Z_e4[:, 1], c=t_e4, cmap="viridis")
plt.colorbar(label="underlying t")
plt.title("Easy 4: 3D cloud compressed to PCA scores")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.show()

▶ What you'll see: the 2D PCA score plot is organized by the underlying one-dimensional trend.

👀 Takeaway: PCA can reveal a low-dimensional coordinate system hidden inside correlated features.

### Easy 5 — Detect unusual reconstruction error

**Goal.** Use PCA residuals as a simple anomaly score, because points that do not fit the learned low-rank structure reconstruct poorly. We build it in 4 steps.

In [ ]:
t_e5 = np.linspace(-2, 2, 25)  # create normal points along a line-like pattern.
normal_e5 = np.column_stack([t_e5, 2 * t_e5 + 0.05 * np.sin(5 * t_e5)])  # mostly one-dimensional normal data.
outlier_e5 = np.array([[0.0, 3.5]])  # add a point far from the learned line.
X_e5 = np.vstack([normal_e5, outlier_e5])  # combine normal data and one unusual point.

print("dataset shape:", X_e5.shape)  # inspect total points.

▶ What you'll see: one extra point is appended to otherwise line-like data.

In [ ]:
mean_e5, V_e5, s_e5, Xc_e5 = pca_fit(normal_e5, r=1)  # fit PCA only on normal structure.
Z_e5 = pca_transform(X_e5, mean_e5, V_e5)  # score all points using the normal PCA lens.
Xhat_e5 = pca_reconstruct(Z_e5, mean_e5, V_e5)  # reconstruct all points from PC1.
resid_e5 = np.linalg.norm(X_e5 - Xhat_e5, axis=1)  # residual distance is anomaly score.

print("largest residual index:", int(np.argmax(resid_e5)), "value:", round(float(resid_e5.max()), 3))

assert int(np.argmax(resid_e5)) == X_e5.shape[0] - 1

In [ ]:
threshold_e5 = resid_e5[:-1].mean() + 3 * resid_e5[:-1].std()  # simple normal-data threshold.

print("threshold:", round(float(threshold_e5), 3), "outlier residual:", round(float(resid_e5[-1]), 3))

assert resid_e5[-1] > threshold_e5

In [ ]:
plt.figure(figsize=(4.5, 3.2))
plt.scatter(X_e5[:-1, 0], X_e5[:-1, 1], color="steelblue", label="normal")
plt.scatter(X_e5[-1:, 0], X_e5[-1:, 1], color="crimson", label="large residual")
plt.plot(Xhat_e5[:, 0], Xhat_e5[:, 1], color="gray", linewidth=1, label="PC1 reconstruction line")
plt.title("Easy 5: PCA reconstruction flags an outlier")
plt.legend(); plt.show()

▶ What you'll see: the red point sits far from the rank-1 PCA line and has the largest residual.

👀 Takeaway: PCA residuals can monitor whether new points fit the learned low-dimensional structure.

## 🔴 Advanced

### Advanced 1 — Compare covariance PCA and correlation PCA

**Goal.** Show how standardization changes PCA on mixed-unit features, because raw covariance and correlation answer different questions. We build it in 4 steps.

In [ ]:
rng_a1 = np.random.default_rng(1)  # local reproducible generator.
x_a1 = np.linspace(0, 10, 40)  # base trend.
X_a1 = np.column_stack([x_a1, 50 + 20 * np.sin(x_a1), 0.5 * x_a1 + 0.1 * rng_a1.normal(size=40)])  # mixed scales.

print("feature stds:", np.round(X_a1.std(axis=0, ddof=1), 3))  # inspect raw units.

▶ What you'll see: the sine feature has a much larger raw standard deviation than the others.

In [ ]:
mean_raw_a1, V_raw_a1, s_raw_a1, Xc_raw_a1 = pca_fit(X_a1, r=1)  # covariance PCA on raw units.
Xz_a1 = (X_a1 - X_a1.mean(axis=0)) / X_a1.std(axis=0, ddof=1)  # correlation PCA via standardization.
mean_z_a1, V_z_a1, s_z_a1, Xc_z_a1 = pca_fit(Xz_a1, r=1)  # PCA after equalizing feature scales.

print("raw PC1 loadings:", np.round(V_raw_a1[:, 0], 3))
print("standardized PC1 loadings:", np.round(V_z_a1[:, 0], 3))

In [ ]:
raw_dominant_a1 = int(np.argmax(np.abs(V_raw_a1[:, 0])))  # which feature dominates raw PC1.
std_dominant_a1 = int(np.argmax(np.abs(V_z_a1[:, 0])))  # which feature dominates standardized PC1.

print("dominant raw feature:", raw_dominant_a1, "dominant standardized feature:", std_dominant_a1)

assert raw_dominant_a1 == 1

In [ ]:
xpos_a1 = np.arange(3)
plt.figure(figsize=(5, 3))
plt.bar(xpos_a1 - 0.18, np.abs(V_raw_a1[:, 0]), width=0.36, label="raw", color="crimson")
plt.bar(xpos_a1 + 0.18, np.abs(V_z_a1[:, 0]), width=0.36, label="standardized", color="seagreen")
plt.xticks(xpos_a1, ["f0", "f1", "f2"]); plt.ylabel("absolute PC1 loading")
plt.title("Advanced 1: covariance vs correlation PCA"); plt.legend(); plt.show()

▶ What you'll see: raw PCA is dominated by the largest-unit feature; standardized PCA spreads attention differently.

👀 Takeaway: choose raw or standardized PCA based on whether units should define importance.

### Advanced 2 — Use PCA whitening to decorrelate scores

**Goal.** Rescale PCA scores to unit variance, because whitening creates uncorrelated components with comparable scale. We build it in 4 steps.

In [ ]:
rng_a2 = np.random.default_rng(2)  # local reproducible generator.
base_a2 = rng_a2.normal(size=200)  # one strong source of variation.
X_a2 = np.column_stack([base_a2, 0.8 * base_a2 + 0.4 * rng_a2.normal(size=200)])  # correlated 2D data.
mean_a2, V_a2, s_a2, Xc_a2 = pca_fit(X_a2)  # fit PCA.
Z_a2 = pca_transform(X_a2, mean_a2, V_a2)  # unwhitened PCA scores.

print("score covariance before whitening:\n", np.round(np.cov(Z_a2, rowvar=False), 3))

▶ What you'll see: PCA scores are already nearly uncorrelated, but their variances are unequal.

In [ ]:
scale_a2 = s_a2 / np.sqrt(X_a2.shape[0] - 1)  # score standard deviation per component.
Zw_a2 = Z_a2 / scale_a2  # whiten by dividing each score column by its standard deviation.
Cwhite_a2 = np.cov(Zw_a2, rowvar=False)  # covariance after whitening.

print("score covariance after whitening:\n", np.round(Cwhite_a2, 3))

assert np.allclose(np.round(Cwhite_a2, 1), np.eye(2))

In [ ]:
print("whitening scales:", np.round(scale_a2, 3))  # inspect how much each PC was divided by.

In [ ]:
plt.figure(figsize=(4.5, 3.5))
plt.scatter(Z_a2[:, 0], Z_a2[:, 1], s=12, alpha=0.5, label="PCA scores")
plt.scatter(Zw_a2[:, 0], Zw_a2[:, 1], s=12, alpha=0.5, label="whitened")
plt.axis("equal"); plt.title("Advanced 2: whitening equalizes component scale")
plt.legend(); plt.show()

▶ What you'll see: whitened scores have a more balanced spread across axes.

👀 Takeaway: whitening keeps PCA directions but removes variance-scale differences between components.

### Advanced 3 — Evaluate rank with held-out reconstruction

**Goal.** Compare ranks on held-out entries, because training reconstruction error alone always improves as rank increases. We build it in 5 steps.

In [ ]:
rows_a3 = np.linspace(-2, 2, 12)  # latent coordinate for rows.
X_full_a3 = np.column_stack([rows_a3, 2 * rows_a3, -rows_a3]) + 0.05 * np.sin(rows_a3)[:, None]  # low-rank-ish full data.
holdout_cols_a3 = np.array([0, 2, 4, 6, 8, 10])  # rows where we will test feature 1 reconstruction.
train_a3 = X_full_a3.copy()  # copy full data for training approximation.
feature_hold_a3 = 1  # hold out the middle feature for selected rows.

print("full shape:", X_full_a3.shape, "held-out count:", len(holdout_cols_a3))

▶ What you'll see: a 12×3 matrix with six held-out coordinates for validation.

In [ ]:
col_mean_a3 = train_a3[:, feature_hold_a3].mean()  # simple imputation value for missing training entries.
train_a3[holdout_cols_a3, feature_hold_a3] = col_mean_a3  # hide validation values using mean fill for this toy demo.
ranks_a3 = [1, 2, 3]  # candidate PCA ranks.
val_rmse_a3 = []  # store held-out reconstruction errors.

print("mean fill used for hidden feature:", round(float(col_mean_a3), 3))

In [ ]:
for r_a3 in ranks_a3:  # fit one PCA reconstruction per rank.
    mean_a3, V_a3, s_a3, Xc_a3 = pca_fit(train_a3, r=r_a3)  # fit PCA to the mean-filled training matrix.
    Z_a3 = pca_transform(train_a3, mean_a3, V_a3)  # compress.
    Xhat_a3 = pca_reconstruct(Z_a3, mean_a3, V_a3)  # reconstruct.
    err_a3 = np.sqrt(np.mean((X_full_a3[holdout_cols_a3, feature_hold_a3] - Xhat_a3[holdout_cols_a3, feature_hold_a3]) ** 2))  # held-out RMSE.
    val_rmse_a3.append(float(err_a3))  # store rank's validation error.

print("validation RMSE by rank:", np.round(val_rmse_a3, 3))

In [ ]:
best_rank_a3 = ranks_a3[int(np.argmin(val_rmse_a3))]  # choose rank with lowest held-out error.

print("best held-out rank:", best_rank_a3)  # inspect selected rank.

assert best_rank_a3 in ranks_a3

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.plot(ranks_a3, val_rmse_a3, marker="o", color="navy")
plt.axvline(best_rank_a3, color="crimson", linestyle="--", label=f"best r={best_rank_a3}")
plt.xticks(ranks_a3); plt.xlabel("PCA rank"); plt.ylabel("held-out RMSE")
plt.title("Advanced 3: validate rank by reconstruction"); plt.legend(); plt.show()

▶ What you'll see: the validation curve selects a rank by out-of-sample reconstruction, not by training error.

👀 Takeaway: held-out reconstruction is a safer rank check than simply maximizing retained training variance.

### Advanced 4 — Detect concept drift with a trained PCA subspace

**Goal.** Compare new batches to an old PCA subspace, because drift can appear as increased reconstruction error or shifted scores. We build it in 4 steps.

In [ ]:
rng_a4 = np.random.default_rng(4)  # local reproducible generator.
t_train_a4 = rng_a4.normal(size=120)  # training latent coordinate.
X_train_a4 = np.column_stack([t_train_a4, 2 * t_train_a4]) + 0.08 * rng_a4.normal(size=(120, 2))  # stable line-like data.
mean_a4, V_a4, s_a4, Xc_a4 = pca_fit(X_train_a4, r=1)  # learn the normal one-dimensional subspace.

print("training PC1:", np.round(V_a4[:, 0], 3))

▶ What you'll see: the learned PC1 follows the stable training line.

In [ ]:
t_new_a4 = rng_a4.normal(size=80)  # new batch latent coordinate.
X_ok_a4 = np.column_stack([t_new_a4, 2 * t_new_a4]) + 0.08 * rng_a4.normal(size=(80, 2))  # non-drift batch.
X_drift_a4 = np.column_stack([t_new_a4, -1.5 * t_new_a4]) + 0.08 * rng_a4.normal(size=(80, 2))  # rotated drift batch.

print("new batch shapes:", X_ok_a4.shape, X_drift_a4.shape)

In [ ]:
def recon_error_a4(X):  # compute mean reconstruction error under the trained rank-1 PCA.
    Z = pca_transform(X, mean_a4, V_a4)  # project into old subspace.
    Xhat = pca_reconstruct(Z, mean_a4, V_a4)  # reconstruct from old subspace.
    return np.linalg.norm(X - Xhat, axis=1)  # per-point residual norms.
err_ok_a4 = recon_error_a4(X_ok_a4)  # residuals for non-drift batch.
err_drift_a4 = recon_error_a4(X_drift_a4)  # residuals for drifted batch.

print("mean residual ok:", round(float(err_ok_a4.mean()), 3), "drift:", round(float(err_drift_a4.mean()), 3))

assert err_drift_a4.mean() > 5 * err_ok_a4.mean()

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.hist(err_ok_a4, bins=18, alpha=0.7, label="same subspace", color="teal")
plt.hist(err_drift_a4, bins=18, alpha=0.7, label="drifted subspace", color="crimson")
plt.title("Advanced 4: PCA residuals reveal drift")
plt.xlabel("rank-1 reconstruction error"); plt.legend(); plt.show()

▶ What you'll see: the drifted batch has much larger residuals under the old PCA line.

👀 Takeaway: a trained PCA subspace can monitor whether future data still follows the same geometry.

### Advanced 5 — Show PCA's linear limitation on curved data

**Goal.** Compress a curved dataset with linear PCA, because PCA can miss structure that is not well approximated by a flat subspace. We build it in 5 steps.

In [ ]:
t_a5 = np.linspace(0, 2 * np.pi, 120)  # angle around a circle.
X_a5 = np.column_stack([np.cos(t_a5), np.sin(t_a5)])  # nonlinear one-dimensional circle in 2D.
mean_a5, Vfull_a5, s_a5, Xc_a5 = pca_fit(X_a5)  # fit full PCA first so explained ratios use all variance.
V_a5 = Vfull_a5[:, :1]  # keep the best linear one-dimensional subspace for reconstruction.

print("explained ratio PC1:", round(float(explained_ratio(s_a5)[0]), 3))

▶ What you'll see: PC1 explains only about half the variance because a circle has no single dominant line.

In [ ]:
Z_a5 = pca_transform(X_a5, mean_a5, V_a5)  # project circle onto a line.
Xhat_a5 = pca_reconstruct(Z_a5, mean_a5, V_a5)  # reconstruct from that line.
err_a5 = np.linalg.norm(X_a5 - Xhat_a5, axis=1)  # reconstruction error around the circle.

print("mean reconstruction error:", round(float(err_a5.mean()), 3), "max:", round(float(err_a5.max()), 3))

assert err_a5.max() > 0.9

In [ ]:
ratio_full_a5 = explained_ratio(s_a5)  # variance split across the two linear axes.

print("full explained ratios:", np.round(ratio_full_a5, 3))  # inspect near-even split.

In [ ]:
plt.figure(figsize=(4.2, 4.2))
plt.scatter(X_a5[:, 0], X_a5[:, 1], s=15, color="steelblue", label="circle")
plt.scatter(Xhat_a5[:, 0], Xhat_a5[:, 1], s=12, color="orange", label="rank-1 PCA reconstruction")
plt.axis("equal"); plt.title("Advanced 5: linear PCA flattens a curve")
plt.legend(); plt.show()

▶ What you'll see: the circular data collapses onto a line, losing the curved geometry.

In [ ]:
plt.figure(figsize=(4.5, 2.6))
plt.plot(t_a5, err_a5, color="crimson")
plt.title("Advanced 5: reconstruction error around the circle")
plt.xlabel("angle"); plt.ylabel("error"); plt.show()

▶ What you'll see: error rises where the circle is far from the chosen PCA line.

👀 Takeaway: PCA is a linear lens; curved manifolds need different tools or additional features.